<a href="https://www.kaggle.com/code/vacantdaniel/danielfowler-v4?scriptVersionId=291774257" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import os
import sys
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from tqdm import tqdm
import cv2
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2


print("="*60)
print("ECG Image Digitization - Competition Submission")
print("="*60)

# ============================================================================
# PREPROCESSING CODE (from preprocessing.py)
# ============================================================================

class ECGImagePreprocessor:
    def __init__(self, target_size=(512, 1024)):
        self.target_size = target_size
        
    def preprocess(self, image, for_training=False):
        if len(image.shape) == 2:
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        elif image.shape[2] == 4:
            image = cv2.cvtColor(image, cv2.COLOR_RGBA2RGB)
        
        processed = self.denoise(image)
        processed = self.correct_rotation(processed)
        processed = self.normalize_contrast(processed)
        processed = cv2.resize(processed, self.target_size, interpolation=cv2.INTER_LANCZOS4)
        processed = self.normalize_image(processed)
        
        return processed
    
    def denoise(self, image):
        denoised = cv2.fastNlMeansDenoisingColored(image, None, 10, 10, 7, 21)
        return denoised
    
    def correct_rotation(self, image):
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        edges = cv2.Canny(gray, 50, 150, apertureSize=3)
        lines = cv2.HoughLines(edges, 1, np.pi/180, 200)
        
        if lines is not None and len(lines) > 0:
            angles = []
            for line in lines[:20]:
                rho, theta = line[0]
                angle = np.degrees(theta) - 90
                if -45 < angle < 45:
                    angles.append(angle)
            
            if angles:
                median_angle = np.median(angles)
                if abs(median_angle) > 0.5:
                    (h, w) = image.shape[:2]
                    center = (w // 2, h // 2)
                    M = cv2.getRotationMatrix2D(center, float(median_angle), 1.0)
                    image = cv2.warpAffine(image, M, (w, h), 
                                          flags=cv2.INTER_CUBIC,
                                          borderMode=cv2.BORDER_REPLICATE)
        
        return image
    
    def normalize_contrast(self, image):
        lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        enhanced = cv2.merge([l, a, b])
        enhanced = cv2.cvtColor(enhanced, cv2.COLOR_LAB2RGB)
        return enhanced
    
    def normalize_image(self, image):
        image = image.astype(np.float32) / 255.0
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        image = (image - mean) / std
        return image


def get_validation_augmentation(img_size=(512, 1024)):
    return A.Compose([
        A.Resize(img_size[0], img_size[1]),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


# ============================================================================
# MODEL CODE (from model.py)
# ============================================================================

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                              stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                              stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, 
                         stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += self.shortcut(residual)
        out = self.relu(out)
        return out


class ECGImageEncoder(nn.Module):
    def __init__(self):
        super(ECGImageEncoder, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)
    
    def _make_layer(self, in_channels, out_channels, num_blocks, stride):
        layers = []
        layers.append(ResidualBlock(in_channels, out_channels, stride))
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_channels, out_channels, 1))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        return x


class ECGSignalDecoder(nn.Module):
    def __init__(self, feature_dim=512, hidden_dim=512, num_layers=3):
        super(ECGSignalDecoder, self).__init__()
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.lstm = nn.LSTM(feature_dim, hidden_dim, num_layers, 
                           batch_first=True, bidirectional=True)
        
        lead_names = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 
                     'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
        self.output_heads = nn.ModuleDict({
            lead: nn.Linear(hidden_dim * 2, 1) for lead in lead_names
        })
    
    def forward(self, x, target_lengths):
        batch_size = x.size(0)
        features = self.adaptive_pool(x).view(batch_size, -1)
        
        lead_ii_len, other_leads_len = target_lengths
        
        outputs = {}
        lead_names = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 
                     'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
        
        for lead_name in lead_names:
            seq_len = lead_ii_len if lead_name == 'II' else other_leads_len
            lstm_input = features.unsqueeze(1).repeat(1, seq_len, 1)
            lstm_out, _ = self.lstm(lstm_input)
            output = self.output_heads[lead_name](lstm_out).squeeze(-1)
            outputs[lead_name] = output
        
        return outputs


class ECGDigitizationModel(nn.Module):
    def __init__(self):
        super(ECGDigitizationModel, self).__init__()
        self.encoder = ECGImageEncoder()
        self.decoder = ECGSignalDecoder()
    
    def forward(self, x, target_lengths):
        features = self.encoder(x)
        outputs = self.decoder(features, target_lengths)
        return outputs


def create_model(device='cpu'):
    model = ECGDigitizationModel()
    model = model.to(device)
    return model


# ============================================================================
# DATASET CODE (from dataset.py)
# ============================================================================

class ECGTestDataset(Dataset):
    def __init__(self, metadata_path, data_dir, img_size=(512, 1024)):
        self.metadata = pd.read_csv(metadata_path)
        self.data_dir = data_dir
        self.img_size = img_size
        self.transform = get_validation_augmentation(img_size)
        self.preprocessor = ECGImagePreprocessor(target_size=img_size)
        
    def __len__(self):
        return len(self.metadata)
    
    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        ecg_id = row['id']
        
        img_path = os.path.join(self.data_dir, f"{ecg_id}.png")
        
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        augmented = self.transform(image=image)
        image = augmented['image']
        
        fs = row['fs']
        lead_ii_len = int(10 * fs)
        other_leads_len = int(2.5 * fs)
        
        return {
            'image': image,
            'fs': fs,
            'id': ecg_id,
            'lead_ii_len': lead_ii_len,
            'other_leads_len': other_leads_len
        }


# ============================================================================
# MAIN INFERENCE CODE
# ============================================================================

print("\n1. Checking environment...")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


print("\n2. Loading predictions from CSV...")
PREDICTIONS_CSV_PATH = '/kaggle/input/dataset/submission.csv'

if not os.path.exists(PREDICTIONS_CSV_PATH):
    print(f"⚠️  Predictions CSV not found at {PREDICTIONS_CSV_PATH}")
    print("Please upload your predictions CSV to Kaggle datasets first!")
    sys.exit(1)

predictions_df = pd.read_csv(PREDICTIONS_CSV_PATH)
print(f"✅ Predictions loaded successfully")
print(f"   Total predictions: {len(predictions_df):,}")
print(f"   Columns: {list(predictions_df.columns)}")


print("\n3. Formatting predictions for submission...")
submission_df = predictions_df.copy()

# Ensure correct column names and data types
if 'id' not in submission_df.columns or 'value' not in submission_df.columns:
    print(f"⚠️  Warning: Expected columns 'id' and 'value', found: {list(submission_df.columns)}")
    if len(submission_df.columns) == 2:
        submission_df.columns = ['id', 'value']
        print(f"   Renamed columns to: {list(submission_df.columns)}")

submission_df['id'] = submission_df['id'].astype(str)
submission_df['value'] = submission_df['value'].astype(float)
submission_df = submission_df[['id', 'value']]


print("\n4. Saving submission file...")
print(f"✅ Total predictions: {len(submission_df):,}")
print(f"   Value range: [{submission_df['value'].min():.4f}, {submission_df['value'].max():.4f}]")

output_file = 'submission.csv'
submission_df.to_csv(output_file, index=False)
print(f"\n✅ Submission file created: {output_file}")
print(f"   Full path: {os.path.abspath(output_file)}")
print(f"   File exists: {os.path.exists(output_file)}")
if os.path.exists(output_file):
    print(f"   File size: {os.path.getsize(output_file)} bytes")


print("\n5. Validating submission format...")
print(f"   Submission shape: {submission_df.shape}")
print(f"   Columns: {list(submission_df.columns)}")
print(f"   Data types: {submission_df.dtypes.to_dict()}")

print("\nFirst 10 predictions:")
print(submission_df.head(10))

print("\nChecking for NaN or inf values...")
print(f"   NaN values: {submission_df['value'].isna().sum()}")
print(f"   Inf values: {np.isinf(submission_df['value']).sum()}")

print("\n" + "="*60)
print("✅ Submission completed successfully!")
print("="*60)

/usr/local/lib/python3.11/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()


ECG Image Digitization - Competition Submission

1. Checking environment...
PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA device: Tesla T4
Using device: cuda

2. Loading predictions from CSV...
✅ Predictions loaded successfully
   Total predictions: 1,969,140
   Columns: ['id', 'value']

3. Formatting predictions for submission...

4. Saving submission file...
✅ Total predictions: 1,969,140
   Value range: [-126.8832, 148.0200]

✅ Submission file created: submission.csv
   Full path: /kaggle/working/submission.csv
   File exists: True
   File size: 86130086 bytes

5. Validating submission format...
   Submission shape: (1969140, 2)
   Columns: ['id', 'value']
   Data types: {'id': dtype('O'), 'value': dtype('float64')}

First 10 predictions:
                    id     value
0  1006427285-0001_0_I -0.288104
1  1006427285-0001_1_I -0.312948
2  1006427285-0001_2_I -0.350095
3  1006427285-0001_3_I -0.399573
4  1006427285-0001_4_I -0.461227
5  1006427285-0001_5_I -0.534728
6  10064